In [1]:
import pandas as pd
import os
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from xgboost import XGBClassifier
import lightgbm as lgb
from imblearn.over_sampling import SMOTE

os.makedirs('models', exist_ok=True)

df = pd.read_csv('data/processed/churn_engineered.csv')

# Keep only numeric columns
df = df.select_dtypes(include='number')

# 1. Prepare X, y
X = df.drop('Churn', axis=1)
y = df['Churn']

# Impute NaNs before SMOTE
imputer = SimpleImputer(strategy='median')
X_imputed = imputer.fit_transform(X)
X = pd.DataFrame(X_imputed, columns=X.columns)

print(f"Shape: {X.shape}, NaNs remaining: {X.isna().sum().sum()}")

# 2. Handle class imbalance with SMOTE
smote = SMOTE(random_state=42)
X_smote, y_smote = smote.fit_resample(X, y)
print(f"Before SMOTE: {y.value_counts().to_dict()}")
print(f"After SMOTE:  {pd.Series(y_smote).value_counts().to_dict()}")

# 3. Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X_smote, y_smote, test_size=0.2, random_state=42, stratify=y_smote
)

# 4. Train models
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, eval_metric='logloss'),
    'LightGBM': lgb.LGBMClassifier(n_estimators=100, max_depth=6, learning_rate=0.1, random_state=42, verbose=-1)
}

trained_models = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    trained_models[name] = model
    print(f"{name} trained ✅")

# Save best model (XGBoost)
import pickle
with open('models/xgb_model.pkl', 'wb') as f:
    pickle.dump(trained_models['XGBoost'], f)
print('\n✅ Model saved to models/xgb_model.pkl')


Shape: (7043, 15), NaNs remaining: 0
Before SMOTE: {0: 5174, 1: 1869}
After SMOTE:  {0: 5174, 1: 5174}
Logistic Regression trained ✅


Random Forest trained ✅


XGBoost trained ✅


LightGBM trained ✅

✅ Model saved to models/xgb_model.pkl
